A set of simulations & hypothesis tests for assessing the statistical power of "minP vs CRE of interest" tests in the shendure dataset...

Essentially a better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-01-26 15:05:24.885506: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 15:05:24.938684: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="16G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [4]:
client.dashboard_link

'http://10.18.22.67:8787/status'

# Ground truth creation

We will use parameter estimates averaged across cell-type nd cre models. 

When we perform the simulation, we are going to IGNORE cell type!

The shendure dataset is characterized by extremely heterogenious transfection, so different sets of CREs are represented in different cell types, likely due to differential clonotype contribution to different cell-types. For this reason, we don't have ground truth values for many combinations of cre_id, cell type. This means that direct simulation runs into problems, since it allows transfection of any cre into any cell type...

We don't care about cell types in this analysis, so we are just going to remove that information and treat the same cre transformed into two different cell-types as two different entities...

See commit `58a65def6964cea9247c15937dd60714489f1750` and 2026-10-23 notes for further discussion.

In [ ]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")
primordial.compute_model_qc()

We use by_cell_type models, expecting that they will have more robust estimates...

In [ ]:
import pandas as pd
import numpy as np

vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
gt_cell_type=pd.concat(vals)

In [ ]:
#casting away from sparse, since it's not sparse anymore
gt_cell_type["mu"] = gt_cell_type["mu"].astype(float)

#this doesn't even work
gt_cell_type["cre_id"]=gt_cell_type["cre_id"].astype(str)
gt_cell_type["cell_type"]=gt_cell_type["cell_type"].astype(str)

In [ ]:
gt_cell_type.dtypes

In [ ]:
gt_cell_type

Now, we discard cell-type information, as discussed above.

In [ ]:
gt_cell_type["cre_id"] = gt_cell_type["cre_id"] + "---" + gt_cell_type["cell_type"]
gt_cell_type["cell_type"] = "reference"
gt_cell_type

In [ ]:
#sanity check
assert len(gt_cell_type) == len(gt_cell_type["cre_id"].unique())
len(gt_cell_type)

Now that we have reasonable mu estimates for the real CRE, let us add 200% "indistinguishable from minP".

In [ ]:
inactive_names=["inactive_"+str(i) for i in range(0,len(gt_cell_type)*2)]
inactive_df=pd.DataFrame({"cre_id":inactive_names})
minP=scm.SHENDURE_BOUNDS.reference_activity
inactive_df["mu"]=minP
inactive_df["cell_type"]="reference"
inactive_df


Then stack with original gt...

In [ ]:
final_gt=pd.concat([gt_cell_type,inactive_df],ignore_index=True).rename({"mu":"true_mean"},axis=1)
final_gt

In [ ]:
#sanity check
assert len(final_gt[["cre_id","cell_type"]].drop_duplicates()) == len(final_gt)

# Creating artificial libraries

In [ ]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [ ]:
libraries[2]

# Creating sim

Create some bounds to simulate from. These will be identical to the normal shendure bounds, except that we simplify to just one cell-type...

In [ ]:
bound=scm.SHENDURE_BOUNDS.copy()

In [ ]:
print(bound.cells_per_cell_type.name)
print(bound.cells_per_cell_type.index.name)
bound.cells_per_cell_type

In [ ]:
type(bound.cells_per_cell_type)

In [ ]:
working=pd.Series({"reference":bound.cells_per_cell_type.sum()})
working.name=bound.cells_per_cell_type.name
working.index.name=bound.cells_per_cell_type.index.name
working

In [ ]:
bound.cells_per_cell_type=working

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-26",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=bound,
                            ground_truth=final_gt)

In [ ]:
sim.gamut()

In [ ]:
sim.save()

# Fit half orthos

In [5]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-26",
                            client=client)

scMPRAforge: INFO: 'state.parquet' found for 'twothird_pow_sim_2026-01-26', loading.


In [6]:
sim.fit_orthos(direction="by_cre")

# Wald precompute: sandwitch

# Wald precompute: opg

# Hypothesis testing

## Add the hypotheses...

In [11]:
example_data=scm.scMPRA_data.from_parquet(sim.scmpradatp/"0.scmpra")

In [13]:
example_data

In [16]:
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=example_data,
    reference_cre="reference",
)

In [18]:
hs_all_ct.df

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Cdk5r1_chr11_12574---ExEndodermVisceral,reference,reference,reference,<NA>
1,Epas1_chr17_10056---ExEndodermVisceral,reference,reference,reference,<NA>
2,Foxa2_chr2_13806---ExEndodermParietal,reference,reference,reference,<NA>
3,Foxa2_chr2_13808---ExEndodermParietal,reference,reference,reference,<NA>
4,Lama1_chr17_7787---ExEndodermParietal,reference,reference,reference,<NA>
...,...,...,...,...,...
4381,Gata4_chr14_5752---SurfaceEctoderm,reference,reference,reference,<NA>
4382,inactive_538,reference,reference,reference,<NA>
4383,inactive_2082,reference,reference,reference,<NA>
4384,inactive_2181,reference,reference,reference,<NA>


In [19]:
sim.add_hypothesis_set("hs_all_ct",hs_all_ct)

In [ ]:
#spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")
#hs_all_cre = scm.make_all_by_cre_hypotheses(
#    counts=demo_counts,
#    reference_cell_type="reference",
#)

# Close the cluster

In [7]:
client.close()
cluster.close()

2026-01-26 14:58:01,258 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('_fit_ortho_helper-56a9134001a64b88213e3620a0148885')" coro=<Worker.execute() done, defined at /home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2026-01-26 14:58:01,258 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('_fit_ortho_helper-dd5a05abfe9f639747d10e996285d2f8')" coro=<Worker.execute() done, defined at /home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2026-01-26 14:58:01,259 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('_fit_ortho_helper-d6b7f666303726261ad350239f9332be')" coro=<Worker.execute() done, defined at /home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker_

In [ ]:
#cells_df=scm.load_df_pickle_debug("permerge_cells_df_2a93a57f821d4c9b95cf54fb6a215508.pkl")
##cells_df=scm.cast_string_keys(cells_df,["cell_type", "cre_id"])
#ground_truth=scm.load_df_pickle_debug("premerge_gt_ad527259c2c743cdaed70e6a02e88a0f.pkl")
##ground_truth=scm.cast_string_keys(ground_truth,["cell_type", "cre_id"])

In [ ]:
#cells_df.merge(ground_truth,
#                on=["cell_type","cre_id"],
#                validate="many_to_one",
#                how="left",
#                indicator=True)